In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import datetime as dt

from pathlib import Path
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

project_dir = Path.cwd()
data_dir = project_dir / 'Raw Data'

In [4]:
# load single-store sales data
sales_long = pd.read_csv(
    data_dir / 'sales_long.csv',
    parse_dates=['date'],
    low_memory=False
)

In [3]:
# recall the level of this dataset
# i.e. a single store with many items over many dates
sales_long.groupby(['store_id','date','cat_id'])[['item_id']].nunique()

item_id
store_id date       cat_id            
CA_1     2011-01-29 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-01-30 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-01-31 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-01 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-02 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-03 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-04 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-05 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-06 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-07 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-08 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-09 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-10 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-11 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-12 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-13 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-14 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-15 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-16 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-17 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-18 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-19 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-20 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-21 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-22 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-23 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-24 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-25 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-26 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-27 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-02-28 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-03-01 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-03-02 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-03-03 FOODS         1437
                    HOBBIES        565
              

# Create inventory summary

In [5]:
# summarize historical demand by product
inventory_summary = (
    sales_long
    .groupby('item_id', observed=True)
    .agg(
        average_daily_demand=('units_sold', 'mean'),
        daily_demand_std=('units_sold', 'std'),
        total_units_sold=('units_sold', 'sum'),
        zero_demand_days=('units_sold', lambda x: (x == 0).sum()),
        days_observed=('units_sold', 'count')
    )
    .reset_index()
)

# calculate the percentage of days with zero demand
inventory_summary['zero_demand_pct'] = (
    inventory_summary['zero_demand_days']
    / inventory_summary['days_observed']
    * 100
)

# rank products by average daily demand
inventory_summary = inventory_summary.sort_values(
    'average_daily_demand',
    ascending=False
).reset_index(drop=True)

# inspect the results
print(f'Products: {inventory_summary["item_id"].nunique():,}')
print(f'Date range: {sales_long["date"].min().date()} to {sales_long["date"].max().date()}')

display(inventory_summary.round(2).head(20))

Products: 3,049
Date range: 2011-01-29 to 2016-04-24
Days observed per product: 1913–1913


,item_id,average_daily_demand,daily_demand_std,total_units_sold,zero_demand_days,days_observed,zero_demand_pct
0,FOODS_3_090,66.49,57.86,127203,469,1913,24.52
1,FOODS_3_586,45.84,16.40,87691,7,1913,0.37
2,FOODS_3_252,39.19,18.72,74971,6,1913,0.31
3,FOODS_3_120,32.36,33.54,61899,667,1913,34.87
4,FOODS_3_714,28.27,14.50,54080,7,1913,0.37
5,FOODS_3_587,27.03,21.45,51717,455,1913,23.78
6,FOODS_3_808,24.10,24.41,46094,385,1913,20.13
7,FOODS_2_197,21.27,12.74,40698,159,1913,8.31
8,FOODS_3_080,20.46,7.76,39145,6,1913,0.31
9,FOODS_3_555,19.80,7.83,37871,6,1913,0.31
